In [ ]:

import os, json, shutil, datetime, itertools, joblib
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import learning_curve
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

In [ ]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------------- PATH SETUP ----------------
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT70_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"
TEST_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/RF_Finetune_Optimized_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- COPY DATA ----------------
def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(SPLIT70_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT70_DIR, "val"), LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copied train/val/test to Colab local.")




✅ Copied train/val/test to Colab local.


In [ ]:
# ---------------- CONFIG ----------------
SEED = 42
IMG_SIZE = (224, 224)
BATCH = 32  # requested
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ---------------- FEATURE EXTRACTOR ----------------
def build_feature_extractor(img_size=IMG_SIZE):
    base = EfficientNetB0(weights="imagenet", include_top=False,
                          input_shape=(img_size[0], img_size[1], 3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    model = Model(inputs=base.input, outputs=x)
    return model, effnet_preprocess

def make_generator(dir_path, preprocess_fn, shuffle=False):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_fn)
    gen = datagen.flow_from_directory(
        dir_path,
        target_size=IMG_SIZE,
        batch_size=BATCH,
        class_mode="sparse",
        shuffle=shuffle,  # keep False for deterministic label order
    )
    return gen

def extract_features(model, generator):
    n = generator.samples
    steps = int(np.ceil(n / BATCH))
    feats, labels = [], []
    for _ in range(steps):
        x_batch, y_batch = next(generator)
        f_batch = model.predict(x_batch, verbose=0)
        feats.append(f_batch)
        labels.append(y_batch)
    X = np.vstack(feats)
    y = np.concatenate(labels).astype(int)
    return X, y

In [ ]:
# ---------------- BUILD & EXTRACT ----------------
feature_model, preprocess_fn = build_feature_extractor(IMG_SIZE)

train_gen = make_generator(LOCAL_TRAIN, preprocess_fn, shuffle=False)
val_gen   = make_generator(LOCAL_VAL,   preprocess_fn, shuffle=False)
test_gen  = make_generator(LOCAL_TEST,  preprocess_fn, shuffle=False)

class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
classes = [idx_to_class[i] for i in range(len(idx_to_class))]

X_train, y_train = extract_features(feature_model, train_gen)
X_val,   y_val   = extract_features(feature_model, val_gen)
X_test,  y_test  = extract_features(feature_model, test_gen)

np.savez(os.path.join(OUT_DIR, "features_train_val_test.npz"),
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test,
         classes=np.array(classes))

with open(os.path.join(OUT_DIR, "class_mapping.json"), "w") as f:
    json.dump({"class_indices": class_indices, "idx_to_class": idx_to_class}, f, indent=2)

print("✅ Feature extraction done. Shapes:",
      X_train.shape, X_val.shape, X_test.shape)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 28000 images belonging to 7 classes.
Found 8616 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
✅ Feature extraction done. Shapes: (28000, 1280) (8616, 1280) (7178, 1280)


In [ ]:
# -------- ExtraTrees (usually faster than RF) --------
from sklearn.ensemble import ExtraTreesClassifier

best_rf = ExtraTreesClassifier(
    n_estimators=300,     # many cheap trees
    max_depth=None,
    max_features="sqrt",
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    bootstrap=False,      # default; keeps it fast
    random_state=SEED,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)

joblib.dump(best_rf, os.path.join(OUT_DIR, "random_forest_best.joblib"))
with open(os.path.join(OUT_DIR, "rf_grid_summary.json"), "w") as f:
    json.dump({"note": "ExtraTrees used (no tuning)"}, f, indent=2)
print("✅ ExtraTrees trained (fast).")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


✅ ExtraTrees trained (fast).


In [ ]:
# ---------------- EVAL HELPERS ----------------
def plot_confmat(cm, classes, title, savepath):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=140)
    im = ax.imshow(cm, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(classes)),
        yticks=np.arange(len(classes)),
        xticklabels=classes,
        yticklabels=classes,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.savefig(savepath, bbox_inches="tight")
    plt.close()

def save_classif_report(y_true, y_pred, classes, prefix):
    report = classification_report(y_true, y_pred, target_names=classes,
                                   output_dict=True, digits=4)
    with open(os.path.join(OUT_DIR, f"{prefix}_classification_report.json"), "w") as f:
        json.dump(report, f, indent=2)
    with open(os.path.join(OUT_DIR, f"{prefix}_classification_report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=classes, digits=4))

def plot_multiclass_roc(y_true, y_score, classes, prefix):
    y_bin = label_binarize(y_true, classes=np.arange(len(classes)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(classes)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # micro-average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_score.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(classes))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(classes)):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(classes)
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    fig, ax = plt.subplots(figsize=(7, 6), dpi=140)
    ax.plot(fpr["micro"], tpr["micro"], label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
    ax.plot(fpr["macro"], tpr["macro"], label=f"macro-average (AUC = {roc_auc['macro']:.3f})")
    for i, cls in enumerate(classes):
        ax.plot(fpr[i], tpr[i], label=f"{cls} (AUC = {roc_auc[i]:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves – {prefix}")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{prefix}_roc_curves.png"), bbox_inches="tight")
    plt.close()

def plot_learning_curve(model, X, y, title, savepath):
    # fewer points => faster
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=3, n_jobs=-1,
        train_sizes=np.linspace(0.2, 1.0, 3),
        scoring="accuracy",
        shuffle=True, random_state=SEED
    )
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)

    plt.figure(figsize=(7, 6), dpi=140)
    plt.plot(train_sizes, train_mean, marker="o", label="Training score")
    plt.plot(train_sizes, val_mean, marker="o", label="CV score")
    plt.xlabel("Training examples")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(savepath, bbox_inches="tight")
    plt.close()

In [ ]:
# ---------------- EVALUATE: TRAIN / VAL / TEST ----------------
for split_name, X, y in [
    ("train", X_train, y_train),
    ("val",   X_val,   y_val),
    ("test",  X_test,  y_test),
]:
    y_pred  = best_rf.predict(X)
    y_proba = best_rf.predict_proba(X)

    save_classif_report(y, y_pred, classes, prefix=f"{split_name}")
    cm = confusion_matrix(y, y_pred)
    plot_confmat(cm, classes, title=f"Confusion Matrix – {split_name.upper()}",
                 savepath=os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png"))
    plot_multiclass_roc(y, y_proba, classes, prefix=split_name)


In [ ]:
# ---------------- SUMMARY ----------------
with open(os.path.join(OUT_DIR, "README.txt"), "w") as f:
    f.write(
f"""Pipeline summary
==================
Backbone: EfficientNetB0 (GAP) -> RandomForest (no tuning; strong preset)
Image size: {IMG_SIZE}
Batch size: {BATCH}
Classes: {classes}
Feature shapes:
  - X_train: {X_train.shape}
  - X_val  : {X_val.shape}
  - X_test : {X_test.shape}

Preset RF:
  n_estimators=220, max_depth=None, max_features='sqrt',
  min_samples_split=2, min_samples_leaf=1, class_weight='balanced'

Artifacts:
- features_train_val_test.npz
- random_forest_best.joblib
- rf_grid_summary.json
- train/val/test classification_report (.json & .txt)
- train/val/test confusion_matrix (.png)
- train/val/test roc_curves (.png)
- learning_curve.png
"""
    )

print("✅ All outputs saved in:", OUT_DIR)

✅ All outputs saved in: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_144317
